In [5]:
def load_data(file_path):
    doc, sentence = [], []
    location, person, organization = [], [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip()=="" or line.startswith("-DOCSTART-"):
                if sentence:
                    doc.append(" ".join([tkn for tkn, lbl in sentence]))
                    for tkn, lbl in sentence:
                        if lbl== "I-ORG":
                            location.append(tkn)
                        elif lbl== "I-PER":
                            person.append(tkn)
                        elif lbl== "I-LOC":
                            organization.append(tkn)
                    sentence = []
                continue
            parts = line.strip().split()
            token = parts[0]
            ner = parts[-1]
            sentence.append((token, ner))
    return doc, person, organization, location

sentences, person, organization, location = load_data("./train.txt")
print(sentences)
print(person)
print(location)

['EU rejects German call to boycott British lamb .', 'Peter Blackburn', 'BRUSSELS 1996-08-22', 'The European Commission said on Thursday it disagreed with German advice to consumers to shun British lamb until scientists determine whether mad cow disease can be transmitted to sheep .', "Germany 's representative to the European Union 's veterinary committee Werner Zwingmann said on Wednesday consumers should buy sheepmeat from countries other than Britain until the scientific advice was clearer .", '" We do n\'t support any such recommendation because we do n\'t see any grounds for it , " the Commission \'s chief spokesman Nikolaus van der Pas told a news briefing .', 'He said further scientific study was required and if it was found that action was needed it should be taken by the European Union .', 'He said a proposal last month by EU Farm Commissioner Franz Fischler to ban sheep brains , spleens and spinal cords from the human and animal food chains was a highly specific and precauti

In [6]:
import spacy
from spacy.matcher import Matcher
from spacy.pipeline import EntityRuler

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

person_pattren = [[{"IS_TITLE": True}, {"IS_TITLE": True}]] 
location_pattren = [[{"IS_TITLE": True}, {"IS_TITLE": True}],
                        [{"IS_UPPER": True}, {"IS_UPPER": True}]] 
org_pattern = [[{"IS_TITLE": True}, {"IS_TITLE": True}]]

matcher.add("PERSON", person_pattren)
matcher.add("LOCATION", location_pattren)
matcher.add("ORG", org_pattern)

def build_entity_ruler(nlp, person, org, locs):
    ruler = nlp.add_pipe("entity_ruler", before="ner")
    pattrens = []

    for p in person: 
        pattrens.append({"label": "PERSON", "pattern": p})
    for o in org: 
        pattrens.append({"label": "ORG", "pattern": o})
    for l in locs: 
        pattrens.append({"label": "LOC", "pattern": l})
    ruler.add_patterns(pattrens)
    return ruler

build_entity_ruler(nlp, person, organization, location)
for text in sentences:
    doc = nlp(text)
    matches = matcher(doc)
    print("Text: ", text)
    
    for ent in doc.ents:
        print("Entity:", ent.text, ent.label_)


Text:  EU rejects German call to boycott British lamb .
Entity: EU ORG
Entity: German LOC
Entity: British LOC
Entity: . LOC
Text:  Peter Blackburn
Entity: Peter PERSON
Entity: Blackburn PERSON
Text:  BRUSSELS 1996-08-22
Entity: BRUSSELS GPE
Entity: 1996-08-22 DATE
Text:  The European Commission said on Thursday it disagreed with German advice to consumers to shun British lamb until scientists determine whether mad cow disease can be transmitted to sheep .
Entity: European LOC
Entity: Commission LOC
Entity: Thursday DATE
Entity: German LOC
Entity: British LOC
Entity: . LOC
Text:  Germany 's representative to the European Union 's veterinary committee Werner Zwingmann said on Wednesday consumers should buy sheepmeat from countries other than Britain until the scientific advice was clearer .
Entity: Germany ORG
Entity: 's ORG
Entity: the PERSON
Entity: European LOC
Entity: Union ORG
Entity: 's LOC
Entity: Werner PERSON
Entity: Zwingmann PERSON
Entity: Wednesday LOC
Entity: Britain GPE
Ent

In [7]:
from spacy import displacy

# Display entities
article = """
Japan secured a lucky win against Germany in the opening match. 
The Prime Minister Fumio Kishida congratulated the national team. 
Meanwhile, Apple Inc announced a new partnership with China to expand its market. 
In France, President Emmanuel Macron praised the collaboration between Google and European companies.
"""
doc = nlp(article)
html = displacy.render(doc, style="ent", jupyter=False)
print(html)
with open("entities.html", "w", encoding="utf-8") as f:
    f.write(html)

<div class="entities" style="line-height: 2.5; direction: ltr"><br>
<mark class="entity" style="background: #ff9561; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    Japan
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align: middle; margin-left: 0.5rem">LOC</span>
</mark>
 secured a lucky win against 
<mark class="entity" style="background: #ff9561; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    Germany
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align: middle; margin-left: 0.5rem">LOC</span>
</mark>
 
<mark class="entity" style="background: #ff9561; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    in
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align: middle; margin-left: 0.5rem">LOC</span>
</mark>
 
<mark

In [8]:
nlp = spacy.load("en_core_web_lg")

for text in sentences:
        doc = nlp(text)
        for ent in doc.ents:
                print("Entity:", ent.text, ent.label_)


Entity: EU ORG
Entity: German NORP
Entity: British NORP
Entity: Peter Blackburn PERSON
Entity: BRUSSELS GPE
Entity: 1996-08-22 DATE
Entity: The European Commission ORG
Entity: Thursday DATE
Entity: German NORP
Entity: British NORP
Entity: Germany GPE
Entity: the European Union 's ORG
Entity: Werner Zwingmann PERSON
Entity: Wednesday DATE
Entity: Britain GPE
Entity: Commission ORG
Entity: Nikolaus van der Pas PERSON
Entity: the European Union ORG
Entity: last month DATE
Entity: EU Farm ORG
Entity: Franz Fischler PERSON
Entity: Fischler PERSON
Entity: EU ORG
Entity: Britain GPE
Entity: France GPE
Entity: BSE ORG
Entity: Fischler PERSON
Entity: EU ORG
Entity: Spanish Farm ORG
Entity: Loyola de Palacio PERSON
Entity: Fischler PERSON
Entity: EU ORG
Entity: France GPE
Entity: Britain GPE
Entity: Fischler PERSON
Entity: EU ORG
Entity: early next month DATE
Entity: BSE ORG
Entity: British NORP
Entity: Thursday DATE
Entity: German NORP
Entity: British NORP
Entity: Europe LOC
Entity: Germany GPE

In [9]:
doc = nlp(article)
html = displacy.render(doc, style="ent", jupyter=False)
print(html)
with open("entities_lg.html", "w", encoding="utf-8") as f:
    f.write(html)

<div class="entities" style="line-height: 2.5; direction: ltr"><br>
<mark class="entity" style="background: #feca74; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    Japan
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align: middle; margin-left: 0.5rem">GPE</span>
</mark>
 secured a lucky win against 
<mark class="entity" style="background: #feca74; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    Germany
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align: middle; margin-left: 0.5rem">GPE</span>
</mark>
 in the opening match. <br>The Prime Minister 
<mark class="entity" style="background: #aa9cfc; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    Fumio Kishida
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align:

In [13]:
nlp.max_length = 2_000_000
doc = " ".join(sentences)

doc = nlp(doc)
html = displacy.render(doc, style="ent", jupyter=False)
print(html)
with open("train_lg.html", "w", encoding="utf-8") as f:
    f.write(html)

<div class="entities" style="line-height: 2.5; direction: ltr">
<mark class="entity" style="background: #7aecec; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    EU
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align: middle; margin-left: 0.5rem">ORG</span>
</mark>
 rejects 
<mark class="entity" style="background: #c887fb; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    German
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align: middle; margin-left: 0.5rem">NORP</span>
</mark>
 call to boycott 
<mark class="entity" style="background: #c887fb; padding: 0.45em 0.6em; margin: 0 0.25em; line-height: 1; border-radius: 0.35em;">
    British
    <span style="font-size: 0.8em; font-weight: bold; line-height: 1; border-radius: 0.35em; vertical-align: middle; margin-left: 0.5rem">NORP</span>
</mark>
 lamb . 
<ma